# *This notebook provides a workflow to apply a simple PageRank algorithm (i.e. without prioritization nodes) on a graph model. The results can then be compared with the ones obtained with a Personalized PageRank (the same threshold to retrieve genes should be applied) in the context of a benchmark for a scientific publication.*

### *Importing the required libraries*

In [ ]:
import sys
sys.path.append("../scripts")

import glob
import io
import json
import omics_analysis
import os
import math
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import re
import seaborn as sns
import shutil
import statistics
import string_analysis
import sys

from sklearn import metrics
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import auc, precision_recall_curve, roc_auc_score
from sklearn.model_selection import KFold
from tqdm import tqdm

### *Reading the training genes and setting up all the input parameters*

In [ ]:
# Setting the process of interest
process = "stalk_cell"

# Reading the training genes
genes = pd.read_csv("../TrainingGenes/training_genes_stalk_cell.csv")
genes = genes["Feature"].to_list()

# Setting input parameters
path_graph = "../graphs/integrated/graph_STRING_CS_100-E-GEOD-45750_corr_07.graphml"
dataset = "STRING_CS_100_E-GEOD-45750_corr_07" 
df = 0.1 #damping factor value
threshold_ranking = 49 # /!\ to ensure a fair comparison this should be the value determined with the PPR model /!\

# Reading the graph
G = nx.read_graphml(path_graph)

# Extracting valid genes
valid_genes = [node for node in G.nodes(data = False) if node in genes]
#valid_genes = [data["name"] for _, data in G.nodes(data = True) if data["name"] in genes] 
print(f"There are {len(valid_genes)} valid genes in this graph")

# Making the proper directories
if not os.path.exists(f"../results/{process}"):
    os.mkdir(f"../results/{process}")

if not os.path.exists(f"../results/{process}/{dataset}"):
    os.mkdir(f"../results/{process}/{dataset}")

if os.path.exists(f"../results/{process}/{dataset}/Simple_PageRank"):
    shutil.rmtree(f"../results/{process}/{dataset}/Simple_PageRank")
    os.mkdir(f"../results/{process}/{dataset}/Simple_PageRank")
    
else:
    os.mkdir(f"../results/{process}/{dataset}/Simple_PageRank")

# Setting the path to the results
path_results = f"../results/{process}/{dataset}/Simple_PageRank"

### *Running the simple PageRank algorithm*

In [ ]:
# Running the simple PageRank
page_rank = nx.pagerank(G, alpha = df, weight = 'weight') 
df_ppr = pd.DataFrame(list(page_rank.items()), columns = ['Genes','PageRank_score']) #We store the result in a Dataframe
df_ppr_sorted = df_ppr.sort_values(by = "PageRank_score", ascending = False, ignore_index = True) #Sorting by descending values to see the best ranked genes first

# Adding a column for labels (if ECs activation : 1 ; else : 0)
df_ppr_sorted["Labels"] = 0
df_ppr_sorted.loc[df_ppr_sorted.Genes.isin(genes), "Labels"] = 1
      
# Saving the final file
df_ppr_sorted.to_csv(f"{path_results}/PageRank_symbols.csv", index = False, sep = ",")

# What's the mean predicted rank of the training genes ?
df_ppr_sorted.index += 1 #We reset the index on 1 instead of 0

ranking = df_ppr_sorted.index[df_ppr_sorted['Genes'].isin(genes)].tolist()
mean_rank = np.mean(ranking)

# Saving the results in a .txt file
with open(f"{path_results}/Average_predicted_rank_for_a_training_gene.txt", "w") as filout:
    filout.write(f"The average predicted rank for a training gene is {mean_rank:.0f} out of {len(df_ppr_sorted['Genes'])}")
    
# What's the AUC ?
X = df_ppr_sorted[["PageRank_score"]]
y = df_ppr_sorted["Labels"]
AUC = round(metrics.roc_auc_score(y, X), 4)

with open(f"{path_results}/AUC.txt", "w") as filout_2:
    filout_2.write(f"AUC : {AUC:.3f}")


# Storing in a .csv file the exact ranking of each of the training genes
training_genes = []
ranks = []
for gene in df_ppr_sorted.itertuples():
    if gene[1] in genes:
        training_genes.append(gene[1])
        ranks.append(gene[0])


ranking = pd.DataFrame({"Training genes": training_genes, "Ranks": ranks})
ranking.to_csv(f"{path_results}/Training_genes_ranks.csv", sep = ",", index = False)

# Reading the PPR ranking file
ranking = df_ppr_sorted

# Using the same threshold to filter out the ranked genes as the one used for the PPR version
ranking_filtered = ranking.head(threshold_ranking)

print(f"Before filtering: {ranking.shape}")
print(f"After filtering: {ranking_filtered.shape}")

# Assessing the enrichment of the best ranked genes
if not os.path.exists(f"{path_results}/Enrichment"):
    os.mkdir(f"{path_results}/Enrichment")
    os.mkdir(f"{path_results}/Enrichment/Best_ranked_genes")
    os.mkdir(f"{path_results}/Enrichment/Training_genes")

else:
    shutil.rmtree(f"{path_results}/Enrichment")
    os.mkdir(f"{path_results}/Enrichment")
    os.mkdir(f"{path_results}/Enrichment/Best_ranked_genes")
    os.mkdir(f"{path_results}/Enrichment/Training_genes")

# Enrichment parameters
enrichr_library = "GO_Biological_Process_2025"
color = "lightskyblue"
threshold_pathways = 10

if threshold_pathways == 10:
    x_param = 24
    y_param = 12

else:
    diff = threshold_pathways - 10
    x_param = 24 + (diff * 1.2)
    y_param = 12 + (diff * 0.6)

# Retrieving the relevant GO annotations for the training genes
results = useful_functions.Enrichr_API(valid_genes, [enrichr_library])
df2 = results[1]
df2.rename(columns = {0: "Index",
                     1: "Term",
                     2: "pvalues",
                     3: "Odds ratio",
                     4: "Combined score",
                     5: "Overlap genes",
                     6: "Adjusted pvalues",
                     7: "unknown1",
                     8: "unknown2"}, inplace = True)

df2 = df2.drop(columns = ["Index", "unknown1", "unknown2"])
df_relevant = df2.loc[df2["Adjusted pvalues"] < 0.05]
df_relevant.to_csv(f"{path_results}/Enrichment/Training_genes/relevant_GO_training_genes.csv",
                  sep = ",", index = False)

# Generating the barplots for the top 10 enriched pathways
plot_name = f"{path_results}/Enrichment/Training_genes/enrichment_top_{threshold_pathways}_pathways_training_genes.png"
results = useful_functions.Enrichr_API_top_n(genes, [enrichr_library], threshold_pathways)
useful_functions.enrichr_figure(results[2], results[3], results[4], plot_name, [enrichr_library], color, threshold_pathways, x_param, y_param)

# Retrieving the relevant GO annotations from the best ranked genes
df_top_ranked_genes = ranking_filtered
gene_list_input = [ ]

for gene in df_top_ranked_genes.itertuples():
    gene_list_input.append(gene[1].upper())

genes = [x.strip() for x in gene_list_input]
results = useful_functions.Enrichr_API(genes, [enrichr_library])

df2 = results[1]
df2.rename(columns = {0:'Index', 1: 'Term', 2: 'pvalues', 
                               3: 'Odds ratio', 4: 'Combined score', 
                               5: 'Overlap genes', 6: 'Adjusted pvalues', 
                               7: 'unknown1', 8: "unknown2"}, inplace = True)
    
df2 = df2.drop(columns = ["Index", "unknown1", "unknown2"])
df_relevant = df2.loc[df2["Adjusted pvalues"] < 0.05]
df_relevant.to_csv(f"{path_results}/Enrichment/Best_ranked_genes/relevant_GO_best_ranked_genes.csv",
                  sep = ",", index = False)

# Generating the barplots for the top 10 enriched pathways
plot_name = f"{path_results}/Enrichment/Best_ranked_genes/enrichment_top_{threshold_pathways}_pathways_best_ranked_genes.png"
results = useful_functions.Enrichr_API_top_n(genes, [enrichr_library], threshold_pathways)
useful_functions.enrichr_figure(results[2], results[3], results[4], plot_name, [enrichr_library], color, threshold_pathways, x_param, y_param) 

# Looking for common relevant GO annotations
TG = []
df_TG = pd.read_csv(f"{path_results}/Enrichment/Training_genes/relevant_GO_training_genes.csv")

for GO in df_TG.itertuples():
    TG.append(GO[1])
    
Best_genes = []
df_best_genes = pd.read_csv(f"{path_results}/Enrichment/Best_ranked_genes/relevant_GO_best_ranked_genes.csv")

for GO in df_best_genes.itertuples():
    Best_genes.append(GO[1])
    
common = []
for GO in Best_genes:
    if GO in TG:
        common.append(GO)
        
ratio = (len(common)/len(TG))*100

print(f"There are {len(common)} GO annotations in common between the best ranked genes and the training genes")
print(f"Enrichment rate : {ratio:.3f} %")

with open(f"{path_results}/Enrichment/Enrichment_info_{dataset}_{process}.txt", "a") as f_out:
    f_out.write(f"There are {len(common)} GO annotations in common between the best ranked genes and the training genes\n")
    f_out.write(f"Enrichment rate : {ratio:.3f} %")


# Computing the Jaccard similarity index and Jaccard dissimilarity distance
reference = pd.read_csv(f"{path_results}/Enrichment/Training_genes/relevant_GO_training_genes.csv")
pathways_reference = set()

for pathway in reference.itertuples():
    pathways_reference.add(pathway[1])

predicted = pd.read_csv(f"{path_results}/Enrichment/Best_ranked_genes/relevant_GO_best_ranked_genes.csv")
pathways_predicted = set()

for pathway in predicted.itertuples():
    pathways_predicted.add(pathway[1])


#Intersection and Union of two sets can also be done using & and | operators
AnB = pathways_reference.intersection(pathways_predicted)
AUB = pathways_reference.union(pathways_predicted)

JAB = float(len(AnB))/float(len(AUB))

#Computing the Jaccard similarity index and Jaccard dissimilarity distance
print(f"Jaccard similarity index J(A,B): {JAB:.3f}")
print(f"Jaccard dissimilarity distance : {(1 - JAB):.3f}")

#Writting the results in the .txt file
with open(f"{path_results}/Enrichment/Enrichment_info_{dataset}_{process}.txt", "a") as f_out:
    f_out.write(f"\nJaccard similarity index : {JAB:.3f}\n")
    f_out.write(f"Jaccard dissimilarity distance : {(1 - JAB):.3f}")

### *Preparing a dedicated subfolder to run a Pubmed Search*

In [ ]:
# Importing the Pubmed search script and aliases file to the PPR folder
if not os.path.exists(f"{path_results}/Aliases.csv"):
    shutil.copyfile("../PubmedSearch/Aliases.csv", f"{path_results}/Aliases.csv")

if not os.path.exists(f"{path_results}/Pubmed_search_Leo_Bettoni.py"):
    shutil.copyfile("../PubmedSearch/Pubmed_search_Leo_Bettoni.py", 
                    f"{path_results}/Pubmed_search_Leo_Bettoni.py")